# 01 — EDA & Preprocessing
## Universal Sequence Lab · Assignment 3

This notebook covers **Exploratory Data Analysis and preprocessing** for all three data modalities used in this project:

| Module | Data | Task |
|--------|------|------|
| **A** | IMDb Reviews (text) | Sentiment Classification |
| **B** | Jena Climate (tabular) | Temperature Forecasting |
| **C** | Multi30k (parallel text) | EN→DE Translation |

---

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────
!pip install datasets sacrebleu seaborn scikit-learn -q

In [ ]:
import sys, os
sys.path.append('/content/src')  # Colab path; adjust if running locally

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Dependencies loaded ✓')

## Module A — IMDb Text Dataset


In [ ]:
from datasets import load_dataset
imdb = load_dataset('imdb')
print(imdb)
print('\nSample review:')
print(imdb['train'][0]['text'][:400])
print('Label:', imdb['train'][0]['label'])

In [ ]:
# ── Label distribution ────────────────────────────────────────────────
labels = [ex['label'] for ex in imdb['train']]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Class balance
axes[0].bar(['Negative (0)', 'Positive (1)'],
            [labels.count(0), labels.count(1)], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Class Distribution (Train)', fontweight='bold')
axes[0].set_ylabel('Count')

# Review length distribution
import re
lengths = [len(re.sub(r'<[^>]+>', ' ', ex['text']).split()) for ex in imdb['train']]
axes[1].hist(lengths, bins=60, color='#3498db', edgecolor='white')
axes[1].axvline(np.median(lengths), color='red', linestyle='--',
                label=f'Median: {int(np.median(lengths))}')
axes[1].set_title('Review Length Distribution (words)', fontweight='bold')
axes[1].set_xlabel('Number of words')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_imdb.png', bbox_inches='tight')
plt.show()
print(f'Median length: {np.median(lengths):.0f} | 95th pct: {np.percentile(lengths, 95):.0f}')

In [ ]:
# ── Vocabulary analysis ───────────────────────────────────────────────
from dataset import simple_tokenize, Vocabulary

vocab = Vocabulary(min_freq=3)
vocab.build([ex['text'] for ex in imdb['train']])

# Most common words
all_tokens = []
for ex in imdb['train']:
    all_tokens.extend(simple_tokenize(ex['text']))
top30 = Counter(all_tokens).most_common(30)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar([w for w, _ in top30], [c for _, c in top30], color='#9b59b6')
ax.set_title('Top 30 Most Frequent Tokens (after basic tokenization)', fontweight='bold')
ax.set_ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'\nVocab size (min_freq=3): {len(vocab):,}')
print('Special tokens:', list(vocab.token2idx.keys())[:4])

## Module B — Jena Climate Time Series


In [ ]:
url = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip'
df = pd.read_csv(url, compression='zip')
df = df.iloc[::6].reset_index(drop=True)  # hourly
df['Date Time'] = pd.to_datetime(df['Date Time'], format='%d.%m.%Y %H:%M:%S')
df = df.set_index('Date Time')

features = ['T (degC)', 'p (mbar)', 'rh (%)', 'wv (m/s)', 'Tdew (degC)']
print(df[features].describe().round(2))
print(f'\nTotal time steps: {len(df):,}')
print(f'Date range: {df.index[0]} → {df.index[-1]}')

In [ ]:
# ── Feature plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(len(features), 1, figsize=(14, 12), sharex=True)
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for ax, feat, color in zip(axes, features, colors):
    ax.plot(df.index[:8760], df[feat].values[:8760],
            linewidth=0.7, color=color, alpha=0.8)
    ax.set_ylabel(feat, fontsize=9)
    ax.grid(alpha=0.2)

axes[0].set_title('Jena Climate — One Year of Features (2009)', fontweight='bold')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('eda_weather.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df[features].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax)
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_weather_corr.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Sliding window explanation ─────────────────────────────────────────
SEQ_LEN, HORIZON = 96, 24
print('Sliding Window Strategy')
print('─' * 40)
print(f'  Input window (X): {SEQ_LEN} hours = {SEQ_LEN} timesteps × {len(features)} features')
print(f'  Forecast horizon (y): next {HORIZON} hours of temperature')
print(f'  Total possible windows: {len(df) - SEQ_LEN - HORIZON + 1:,}')

## Module C — Multi30k EN→DE Translation


In [ ]:
multi30k = load_dataset('bentrevett/multi30k')
print(multi30k)
print('\nSample pairs:')
for ex in list(multi30k['train'])[:5]:
    print(f'  EN: {ex["en"]}')
    print(f'  DE: {ex["de"]}')
    print()

In [ ]:
# ── Length analysis ────────────────────────────────────────────────────
en_lens = [len(ex['en'].split()) for ex in multi30k['train']]
de_lens = [len(ex['de'].split()) for ex in multi30k['train']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, lens, lang, color in zip(axes,
    [en_lens, de_lens], ['English', 'German'], ['#3498db', '#e67e22']):
    ax.hist(lens, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(lens), color='black', linestyle='--',
               label=f'Mean: {np.mean(lens):.1f}')
    ax.set_title(f'{lang} Sentence Lengths', fontweight='bold')
    ax.set_xlabel('Tokens')
    ax.legend()

plt.tight_layout()
plt.savefig('eda_translation.png', bbox_inches='tight')
plt.show()

print(f'EN  — mean: {np.mean(en_lens):.1f}, max: {max(en_lens)}, 95th pct: {np.percentile(en_lens,95):.0f}')
print(f'DE  — mean: {np.mean(de_lens):.1f}, max: {max(de_lens)}, 95th pct: {np.percentile(de_lens,95):.0f}')

# Length ratio
ratio = np.array(de_lens) / np.array(en_lens)
print(f'DE/EN length ratio: {ratio.mean():.3f} ± {ratio.std():.3f}')

In [ ]:
# ── Vocabulary overlap ─────────────────────────────────────────────────
en_counter = Counter(t for ex in multi30k['train'] for t in ex['en'].lower().split())
de_counter = Counter(t for ex in multi30k['train'] for t in ex['de'].lower().split())
print(f'Unique EN tokens: {len(en_counter):,}')
print(f'Unique DE tokens: {len(de_counter):,}')
print(f'Top 10 EN: {en_counter.most_common(10)}')
print(f'Top 10 DE: {de_counter.most_common(10)}')

## Summary — Preprocessing Decisions

| Module | Key Decision | Rationale |
|--------|-------------|----------|
| **A (NLP)** | max_len=256, min_freq=3, vocab=~25K | Covers 95th pct of review lengths |
| **A (NLP)** | Bidirectional LSTM/GRU | Reviews benefit from full context |
| **B (TS)** | seq_len=96h, horizon=24h | Capture daily patterns; predict 1 day ahead |
| **B (TS)** | StandardScaler on train only | Prevent data leakage |
| **C (Seq2Seq)** | max_len=50, min_freq=2 | Most sentences < 40 tokens |
| **C (Seq2Seq)** | Teacher forcing ratio=0.5 | Balance stability and generalization |
